# Full-pass training: staged resume and reduced notebook output

Use this notebook for the **existing** `fullpass_both_t4_v1` experiment. Stop its other training cell before starting this notebook; never run two copies of the same experiment at once.

1. Run Cell 1 to mount Drive, authenticate, and verify the saved training source.
2. Optionally run Cell 2 to inspect the reported checkpoint and available system RAM.
3. Run Cell 3 for **two additional completed epochs of one model**, then let the child processes exit. Rerun Cell 3 for the next stage. Finish YOLO at epoch 100, then set `ACTIVE_MODEL="unet"`.

The architecture, frozen data split, full-image coverage, optimizer, 100-epoch schedule, and existing checkpoint identity are unchanged. No source files in the saved experiment are patched. A stale state mirror can request a stage already completed on the server; the worker verifies the server state and skips completed work.

This limits the lifetime of each training process and throttles repetitive batch output. It is not a diagnosis or guarantee against RAM exhaustion. The RAM report distinguishes available host memory from the summed resident memory of the training processes. GPU memory is separate. Reports are printed when worker output arrives.

If RAM is almost exhausted, stop rather than waiting for an out-of-memory crash. YOLO resumes from the last **server-verified completed epoch**; an interrupted epoch is replayed. A successful save prints `YOLO checkpoint confirmed on Drive: epoch ...`. After restarting the runtime, run Cell 1 again before Cell 3. Drive checkpoints persist; a replaced VM may need its local disk caches rebuilt.

Full worker logs remain in `OUTPUT/training.log`; this notebook prints at most one repetitive YOLO batch line per 30 seconds. Clearing old notebook output can reduce notebook/browser load, but it does not free tensors owned by a running training process. Running another cell alone does not reset RAM.


In [ ]:
# Cell 1: reconnect and verify the existing training source; no training starts here.
from pathlib import Path
from google.colab import auth, drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
OUTPUT = BASE / "fixed_split_experiments/fullpass_both_t4_v1"
CHASE_TRAINING_ROOT = BASE / "fixed_split_experiments/chase_shape_best84_recipe_t4_v1"
CACHE_ROOT = Path("/content/fullpass_both_t4_cache")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
CHECKOUT = Path("/content/solar_fullpass_both_github")
drive.mount("/content/drive")
# Drive API verification is required before a checkpoint is committed.
auth.authenticate_user()
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
state_path = OUTPUT / "bootstrap_state.json"
if not state_path.is_file():
    raise FileNotFoundError("Existing full-pass run not found. Check the Drive account and OUTPUT path.")
if state_path.is_file():
    state = json.loads(state_path.read_text())
    if state.get("scope") != "fullpass_both_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("Saved bootstrap belongs to another experiment.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed; existing files were preserved.")
    manifest = json.loads(manifest_path.read_text())
    identity = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(identity).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    for name, expected in manifest.items():
        path = snapshot / name
        if path.is_symlink() or not path.resolve().is_relative_to(snapshot.resolve()):
            raise ValueError("Unsafe saved source path.")
        if not path.is_file() or sha(path) != expected:
            raise ValueError("Saved launcher/source changed: " + name)
    bootstrap = snapshot / "tools/fullpass_both_colab.py"
    if "tools/fullpass_both_colab.py" not in manifest:
        raise ValueError("Saved manifest does not contain the launcher.")
print("Verified existing source:", version)
print("Experiment:", OUTPUT)


In [ ]:
# Cell 2: optional status check. Drive state files are reporting mirrors.
# The training worker always verifies the authoritative server checkpoint on resume.
for model_name in ("yolo", "unet"):
    path = OUTPUT / model_name / "state.json"
    if path.is_file():
        record = json.loads(path.read_text())
        print(model_name, "reported completed epoch:", record.get("epoch"),
              "best epoch:", record.get("best_epoch"), "best PQ:", record.get("best_pq"))
    else:
        print(model_name, "no local state mirror yet")
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM available: {vm.available / 2**30:.2f} GiB / {vm.total / 2**30:.2f} GiB")
except ImportError:
    print("Use Colab's Resources panel to inspect system RAM.")


In [ ]:
# Cell 3: run one model for a short stage, then release the child processes.
# Rerun this cell for the next stage. After a runtime restart, rerun Cell 1 first.
# Keep yolo until epoch 100, then change ACTIVE_MODEL to "unet".
ACTIVE_MODEL = "yolo"
STAGE_EPOCHS = 2
import time
if ACTIVE_MODEL not in ("yolo", "unet") or type(STAGE_EPOCHS) is not int or not 1 <= STAGE_EPOCHS <= 100:
    raise ValueError("Choose yolo/unet and a positive stage length up to 100.")
state_file = OUTPUT / ACTIVE_MODEL / "state.json"
reported = json.loads(state_file.read_text()) if state_file.is_file() else {}
completed = reported.get("epoch", 0)
if type(completed) is not int or not 0 <= completed <= 100:
    raise ValueError("Invalid state mirror. Preserve it and inspect the verified checkpoint.")
run_until = min(100, completed + STAGE_EPOCHS)
print(f"{ACTIVE_MODEL}: reported completed {completed}; requested stage ends at {run_until}/100.")
print("The 100-epoch learning-rate schedule and all 5,928 tiles per epoch are unchanged.")
print("Resume starts from the server-verified checkpoint, not from this reporting mirror.")
command = [sys.executable, "-u", str(bootstrap), "--base", str(BASE),
           "--output", str(OUTPUT), "--cache-root", str(CACHE_ROOT),
           "--chase-training-root", str(CHASE_TRAINING_ROOT),
           "--phase", "all", "--run-until", str(run_until), "--models", ACTIVE_MODEL]
process = subprocess.Popen(command, start_new_session=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", bufsize=1)
last_progress = last_memory = 0.0
progress_line = re.compile(r"^\s*\d+/\d+\s+")
try:
    import psutil
except ImportError:
    psutil = None
try:
    for line in process.stdout:
        now = time.monotonic()
        clean = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", line)
        if not clean.strip():
            continue
        # The original worker still writes every line to OUTPUT/training.log.
        # Only repetitive YOLO batch lines are throttled; errors/save messages remain visible.
        if not progress_line.match(clean) or now - last_progress >= 30:
            print(line, end="", flush=True)
            if progress_line.match(clean):
                last_progress = now
        if psutil is not None and now - last_memory >= 60:
            vm = psutil.virtual_memory()
            try:
                parent = psutil.Process(process.pid)
                members = [parent] + parent.children(recursive=True)
                rss = 0
                for member in members:
                    try:
                        rss += member.memory_info().rss
                    except (psutil.NoSuchProcess, psutil.AccessDenied):
                        pass
                print(f"[RAM] available {vm.available / 2**30:.2f} GiB; "
                      f"training process tree summed RSS {rss / 2**30:.2f} GiB "
                      "(shared pages may be counted more than once)", flush=True)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
            if vm.available < 700 * 1024**2:
                print("[RAM] Less than 700 MiB available. If still falling, stop the cell; "
                      "the unfinished YOLO epoch will replay from the last verified save.", flush=True)
            last_memory = now
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
    process.stdout.close()
result_path = OUTPUT / ACTIVE_MODEL / "result.json"
if result_path.is_file():
    result = json.loads(result_path.read_text())
    print("Stage finished:", ACTIVE_MODEL, result.get("status"),
          "completed epoch:", result.get("epochs_completed"))
print("Child processes exited. Rerun Cell 3 for the next stage.")
print("If system RAM remains high, restart the Colab session and rerun Cells 1 and 3.")
print("Review files stay in:", OUTPUT)
